# SelectionEarthquake full feature walkthrough

This notebook demonstrates the main public features of the `earthquake-selection` package using the real Python import package `selection_service`.

It is designed to run offline with the packaged PEER flatfile. AFAD search and waveform download cells are included behind a `RUN_AFAD = False` flag because they require network access and live AFAD availability.

## 1. Environment setup

If the package is installed with `pip install earthquake-selection`, the import below works directly. If this notebook is opened from the repository checkout, the `src` directory is added to `sys.path` for local development.

In [1]:
from pathlib import Path
import json
import sys

repo_root = Path.cwd()
if (repo_root / "src").exists():
    sys.path.insert(0, str(repo_root / "src"))
elif (repo_root.parent / "src").exists():
    sys.path.insert(0, str(repo_root.parent / "src"))

import pandas as pd
from IPython.display import display
import selection_service

from selection_service import (
    EarthquakeAPI,
    DesignCode,
    ProviderName,
    SelectionConfig,
    SearchCriteria,
    TBDYSelectionStrategy,
    TBDY2018ConstraintStrategy,
    ConstraintSelectionStrategy,
    ParetoSelectionStrategy,
    SpectrumMatchStrategy,
    setup_logging,
)
from selection_service.processing.Selection import ScoringWeights

setup_logging()
selection_service.__version__

'1.2.0'

## 2. Selection configuration

`SelectionConfig` controls record count, diversity limits, minimum score, and design code.

In [2]:
config = SelectionConfig(
    design_code=DesignCode.TBDY_2018,
    num_records=11,
    max_per_station=3,
    max_per_event=3,
    min_score=55.0,
)

config.model_dump()

{'design_code': <DesignCode.TBDY_2018: 'TBDY_2018'>,
 'num_records': 11,
 'max_per_station': 3,
 'max_per_event': 3,
 'min_score': 55.0,
 'required_components': []}

## 3. Scoring presets and custom weights

The library ships documented scoring presets. You can also override any weight explicitly.

In [3]:
ScoringWeights.preset_descriptions()

{'balanced': 'Default preset. Magnitude, distance, site class, intensity, duration, depth, and mechanism all contribute to the score.',
 'tbdy_2018_record_selection': 'TBDY 2018-oriented preset. Emphasizes magnitude, source-to-site distance, Vs30, and fault mechanism for record selection.',
 'site_response': 'Site-response preset. Gives more influence to Vs30, duration, and intensity measures while keeping magnitude and distance active.'}

In [4]:
tbdy_weights = ScoringWeights.from_preset("tbdy_2018_record_selection")
site_response_weights = ScoringWeights.from_preset("site_response")
custom_weights = ScoringWeights(
    magnitude=6.0,
    rjb=5.0,
    rrup=5.0,
    vs30=5.0,
    mechanism=4.0,
    pga=2.0,
    pgv=2.0,
)

pd.DataFrame([
    {"preset": "tbdy_2018_record_selection", **tbdy_weights.model_dump()},
    {"preset": "site_response", **site_response_weights.model_dump()},
    {"preset": "custom", **custom_weights.model_dump()},
]).set_index("preset")

,magnitude,rjb,rrup,repi,vs30,pga,pgv,pgd,t90,arias,depth,mechanism
preset,,,,,,,,,,,,
tbdy_2018_record_selection,6.0,5.0,5.0,3.0,5.0,3.0,2.5,1.5,2.0,1.5,1.0,4.0
site_response,3.0,3.0,3.0,2.0,6.0,4.0,4.0,3.0,5.0,4.0,1.0,2.0
custom,6.0,5.0,5.0,4.0,5.0,2.0,2.0,2.5,3.0,2.0,2.0,4.0


## 4. Search criteria and provider parameter conversion

`SearchCriteria` is provider-neutral. It can be converted into PEER, AFAD, or FDSN-style parameters. The same criteria also define targets used by scoring and error metrics.

In [5]:
criteria = SearchCriteria(
    start_date="2000-01-01",
    end_date="2025-09-05",
    min_magnitude=7.0,
    max_magnitude=8.0,
    min_vs30=300.0,
    max_vs30=400.0,
    min_Rjb=0.0,
    max_Rjb=100.0,
    min_Rrup=0.0,
    max_Rrup=120.0,
    min_pga=50.0,
    max_pga=500.0,
    min_pgv=5.0,
    max_pgv=100.0,
    mechanisms=["StrikeSlip"],
    weights=tbdy_weights,
)

{
    "target_magnitude": criteria.get_effective_target("magnitude"),
    "target_rjb": criteria.get_effective_target("rjb"),
    "target_vs30": criteria.get_effective_target("vs30"),
    "mechanism_targets": criteria.get_mechanism_targets(),
}

{'target_magnitude': 7.5,
 'target_rjb': 50.0,
 'target_vs30': 350.0,
 'mechanism_targets': ['StrikeSlip']}

In [6]:
print("PEER params")
display(criteria.to_peer_params())

print("AFAD params")
display(criteria.to_afad_params())

print("FDSN params")
display(criteria.to_fdsn_params())

PEER params


{'year_start': 2000,
 'year_end': 2025,
 'min_magnitude': 7.0,
 'max_magnitude': 8.0,
 'min_vs30': 300.0,
 'max_vs30': 400.0,
 'min_Rjb': 0.0,
 'max_Rjb': 100.0,
 'min_Rrup': 0.0,
 'max_Rrup': 120.0,
 'min_depth': None,
 'max_depth': None,
 'min_pga': 50.0,
 'max_pga': 500.0,
 'min_pgv': 5.0,
 'max_pgv': 100.0,
 'min_pgd': None,
 'max_pgd': None,
 'mechanisms': [0]}

AFAD params


{'startDate': '2000-01-01T00:00:00.000Z',
 'endDate': '2025-09-05T23:59:59.999Z',
 'fromMagnitude': 7.0,
 'toMagnitude': 8.0,
 'fromRjb': 0.0,
 'toRjb': 100.0,
 'fromRrup': 0.0,
 'toRrup': 120.0,
 'fromVs30': 300.0,
 'toVs30': 400.0,
 'fromPGA': 50.0,
 'toPGA': 500.0,
 'fromPGV': 5.0,
 'toPGV': 100.0,
 'faultType': 'SS'}

FDSN params


{'starttime': '2000-01-01T00:00:00.000Z',
 'endtime': '2025-09-05T23:59:59.999Z',
 'minmagnitude': 7.0,
 'maxmagnitude': 8.0,
 'latitude': None,
 'longitude': None}

In [7]:
# Validation example: invalid ranges fail early.
try:
    SearchCriteria(
        start_date="2025-01-01",
        end_date="2024-01-01",
        min_magnitude=8.0,
        max_magnitude=7.0,
    )
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc).splitlines()[0])

ValidationError
1 validation error for SearchCriteria


## 5. Strategies

The notebook runs all current strategy families on the local PEER provider:

- `TBDYSelectionStrategy`: weighted Gaussian score.
- `TBDY2018ConstraintStrategy`: hard filters, error metrics, and diversity-controlled selection.
- `ConstraintSelectionStrategy`: backward-compatible alias for `TBDY2018ConstraintStrategy`.
- `ParetoSelectionStrategy`: nondominated candidate ranking.
- `SpectrumMatchStrategy`: spectrum/intensity proxy matching using PGA, PGV, PGD, Arias, and T90 when available.

In [8]:
strategy_classes = {
    "gaussian": TBDYSelectionStrategy,
    "constraint": TBDY2018ConstraintStrategy,
    "constraint_alias": ConstraintSelectionStrategy,
    "pareto": ParetoSelectionStrategy,
    "spectrum": SpectrumMatchStrategy,
}

strategies = {
    name: strategy_class(config=config)
    for name, strategy_class in strategy_classes.items()
}

{name: strategy.get_name() for name, strategy in strategies.items()}

{'gaussian': 'TBDY_2018_Gaussian',
 'constraint': 'TBDY_2018_Constraint',
 'constraint_alias': 'TBDY_2018_Constraint',
 'pareto': 'Pareto_Selection',
 'spectrum': 'Spectrum_Match'}

## 6. End-to-end local PEER flow

This is the core offline workflow: search, select, report, and inspect selected records.

In [9]:
def run_peer_selection(strategy):
    api = EarthquakeAPI(
        provider_names=[ProviderName.PEER],
        strategies=[strategy],
        use_cache=True,
    )
    result = api.run_sync(criteria=criteria, strategy_name=strategy.get_name())
    if not result.success:
        raise RuntimeError(result.error)
    return result.value

peer_results = {
    name: run_peer_selection(strategy)
    for name, strategy in strategies.items()
}

pd.DataFrame([
    {
        "strategy_key": name,
        "strategy_name": result.report["strategy"],
        "selected_count": len(result.selected_df),
        "total_considered": result.report["total_considered"],
        "status": result.report["status"],
    }
    for name, result in peer_results.items()
])

,strategy_key,strategy_name,selected_count,total_considered,status
0,gaussian,TBDY_2018_Gaussian,11,59,success
1,constraint,TBDY_2018_Constraint,11,59,success
2,constraint_alias,TBDY_2018_Constraint,11,59,success
3,pareto,Pareto_Selection,11,59,success
4,spectrum,Spectrum_Match,11,59,success


In [10]:
result = peer_results["constraint"]
selected = result.selected_df
scored = result.scored_df

display_columns = [
    "PROVIDER",
    "RSN",
    "EVENT",
    "MAGNITUDE",
    "STATION",
    "VS30(m/s)",
    "RJB(km)",
    "MECHANISM",
    "SCORE",
    "ERROR_TOTAL",
    "SELECTION_STATUS",
    "SELECTION_REASON",
]

selected[[column for column in display_columns if column in selected.columns]].head(11)

,PROVIDER,RSN,EVENT,MAGNITUDE,STATION,VS30(m/s),RJB(km),MECHANISM,SCORE,ERROR_TOTAL,SELECTION_STATUS,SELECTION_REASON
22,PEER,1163,"Kocaeli, Turkey",7.51,Hava Alani,354.37,58.33,StrikeSlip,88.544542,0.129375,selected,selected
24,PEER,1177,"Kocaeli, Turkey",7.51,Zeytinburnu,341.56,51.98,StrikeSlip,88.292310,0.132601,selected,selected
19,PEER,1157,"Kocaeli, Turkey",7.51,Cekmece,346.00,64.95,StrikeSlip,88.204623,0.133727,selected,selected
16,PEER,900,Landers,7.28,Yermo Fire Station,353.63,23.62,StrikeSlip,86.641802,0.154177,selected,selected
10,PEER,862,Landers,7.28,Indio - Coachella Canal,339.02,54.25,StrikeSlip,85.198116,0.173735,selected,selected
6,PEER,848,Landers,7.28,Coolwater,352.98,19.74,StrikeSlip,84.906395,0.177768,selected,selected
27,PEER,1636,"Manjil, Iran",7.37,Qazvin,302.64,49.97,StrikeSlip,83.949796,0.191188,selected,selected
26,PEER,1634,"Manjil, Iran",7.37,Abhar,302.64,75.58,StrikeSlip,83.926419,0.191520,selected,selected
17,PEER,1144,Gulf of Aqaba,7.20,Eilat,354.88,43.29,StrikeSlip,83.610799,0.196018,selected,selected
28,PEER,1762,Hector Mine,7.13,Amboy,382.93,41.81,StrikeSlip,83.042677,0.204200,selected,selected


## 7. Traceability report

The report explains selected/rejected counts, score breakdowns, and error metrics. The full `scored_df` also contains per-record reasons.

In [11]:
report = result.report
report.keys()

dict_keys(['status', 'search_criteria', 'selected_count', 'total_considered', 'strategy', 'providers', 'records', 'statistics', 'selection_summary', 'score_breakdown', 'error_metrics'])

In [12]:
report["selection_summary"]

{'status_counts': {'rejected': 48, 'selected': 11},
 'rejection_reasons': {'num_records_limit:11': 41, 'max_per_event:3': 7}}

In [13]:
# Why records were rejected or selected.
scored["SELECTION_REASON"].value_counts().head(10)

SELECTION_REASON
num_records_limit:11    41
selected                11
max_per_event:3          7
Name: count, dtype: int64

In [14]:
# Constraint strategy hard filters and error metrics for the first selected record.
first = selected.iloc[0]
display(pd.DataFrame(first["HARD_FILTERS"]))
display(pd.DataFrame(first["ERROR_METRICS"]))

,criterion,column,status,value,min,max,reason,target
0,magnitude,MAGNITUDE,passed,7.51,7.0,8.0,,NaN
1,rjb,RJB(km),passed,58.33,0.0,100.0,,NaN
2,rrup,RRUP(km),passed,60.05,0.0,120.0,,NaN
3,vs30,VS30(m/s),passed,354.37,300.0,400.0,,NaN
4,pga,PGA(cm2/sec),passed,87.990167,50.0,500.0,,NaN
5,pgv,PGV(cm/sec),passed,19.0,5.0,100.0,,NaN
6,mechanism,MECHANISM,passed,StrikeSlip,NaN,NaN,,[StrikeSlip]


,criterion,column,status,target,value,absolute_error,normalized_error,scale,match
0,magnitude,MAGNITUDE,active,7.5,7.51,0.010000,0.010000,1.0,NaN
1,rjb,RJB(km),active,50.0,58.33,8.330000,0.083300,100.0,NaN
2,rrup,RRUP(km),active,60.0,60.05,0.050000,0.000417,120.0,NaN
3,vs30,VS30(m/s),active,350.0,354.37,4.370000,0.043700,100.0,NaN
4,pga,PGA(cm2/sec),active,275.0,87.990167,187.009833,0.415577,450.0,NaN
5,pgv,PGV(cm/sec),active,52.5,19.0,33.500000,0.352632,95.0,NaN
6,mechanism,MECHANISM,active,[StrikeSlip],StrikeSlip,0.000000,0.000000,NaN,1.0


In [15]:
# Gaussian strategy stores weighted scoring contribution in SCORE_BREAKDOWN.
gaussian_result = peer_results["gaussian"]
pd.DataFrame(gaussian_result.selected_df.iloc[0]["SCORE_BREAKDOWN"])

,criterion,column,status,target,value,weight,sigma,raw_score,weighted_score
0,magnitude,MAGNITUDE,active,7.5,7.51,6.0,0.250000,0.999200,5.995202
1,rjb,RJB(km),active,50.0,64.95,5.0,33.333333,0.904316,4.521581
2,rrup,RRUP(km),active,60.0,66.69,5.0,40.000000,0.986111,4.930555
3,vs30,VS30(m/s),active,350.0,346.0,5.0,20.000000,0.980199,4.900993
4,pga,PGA(cm2/sec),active,275.0,156.053221,3.0,112.500000,0.571811,1.715433
5,pgv,PGV(cm/sec),active,52.5,12.931,2.5,23.750000,0.249603,0.624008
6,mechanism,MECHANISM,active,[StrikeSlip],StrikeSlip,4.0,NaN,1.000000,4.000000


## 8. Pareto and spectrum-specific outputs

In [16]:
pareto = peer_results["pareto"].scored_df
pareto_columns = ["RSN", "EVENT", "SCORE", "ERROR_TOTAL", "PARETO_RANK", "PARETO_FRONT", "SELECTION_STATUS", "SELECTION_REASON"]
pareto[[column for column in pareto_columns if column in pareto.columns]].sort_values(
    ["PARETO_RANK", "ERROR_TOTAL"], ascending=[True, True]
).head(15)

,RSN,EVENT,SCORE,ERROR_TOTAL,PARETO_RANK,PARETO_FRONT,SELECTION_STATUS,SELECTION_REASON
22,1163,"Kocaeli, Turkey",88.544542,0.129375,0,True,selected,selected
24,1177,"Kocaeli, Turkey",88.292310,0.132601,0,True,selected,selected
19,1157,"Kocaeli, Turkey",88.204623,0.133727,0,True,selected,selected
20,1160,"Kocaeli, Turkey",87.069038,0.148514,0,True,rejected,max_per_event:3
16,900,Landers,86.641802,0.154177,0,True,selected,selected
21,1162,"Kocaeli, Turkey",85.517536,0.169351,0,True,rejected,max_per_event:3
18,1149,"Kocaeli, Turkey",85.350617,0.171638,0,True,rejected,max_per_event:3
10,862,Landers,85.198116,0.173735,0,True,selected,selected
6,848,Landers,84.906395,0.177768,0,True,selected,selected
46,3756,Landers,84.649853,0.181337,0,True,rejected,max_per_event:3


In [17]:
spectrum = peer_results["spectrum"].scored_df
spectrum_columns = ["RSN", "EVENT", "SCORE", "ERROR_TOTAL", "SPECTRUM_ERROR", "SELECTION_STATUS", "SELECTION_REASON"]
spectrum[[column for column in spectrum_columns if column in spectrum.columns]].sort_values(
    ["SPECTRUM_ERROR", "ERROR_TOTAL"], ascending=[True, True]
).head(15)

,RSN,EVENT,SCORE,ERROR_TOTAL,SPECTRUM_ERROR,SELECTION_STATUS,SELECTION_REASON
11,864,Landers,89.827294,0.113247,0.113247,selected,selected
16,900,Landers,88.653490,0.127987,0.127987,selected,selected
26,1634,"Manjil, Iran",85.055492,0.175703,0.175703,selected,selected
6,848,Landers,84.872599,0.178237,0.178237,selected,selected
13,881,Landers,80.376504,0.244145,0.244145,rejected,max_per_event:3
45,3753,Landers,80.142714,0.247774,0.247774,rejected,max_per_event:3
28,1762,Hector Mine,79.778195,0.253475,0.253475,selected,selected
44,2114,"Denali, Alaska",78.935700,0.266854,0.266854,selected,selected
53,6893,"Darfield, New Zealand",78.156275,0.279488,0.279488,selected,selected
25,1615,"Duzce, Turkey",78.143995,0.279689,0.279689,selected,selected


## 9. Re-selection on an existing dataframe

`EarthquakeAPI.re_selection` applies a different strategy or criteria to an existing dataframe without fetching provider data again.

In [18]:
reselect_criteria = criteria.model_copy(update={
    "min_vs30": 250.0,
    "max_vs30": 500.0,
    "target_vs30": 360.0,
})

pareto_strategy = strategies["pareto"]
api = EarthquakeAPI(
    provider_names=[ProviderName.PEER],
    strategies=[pareto_strategy],
    use_cache=True,
)
reselected = api.re_selection(
    df=gaussian_result.scored_df,
    strategy_name=pareto_strategy.get_name(),
    new_criteria=reselect_criteria,
)

if not reselected.success:
    raise RuntimeError(reselected.error)

reselected.value.selected_df[["RSN", "EVENT", "SCORE", "ERROR_TOTAL", "PARETO_RANK", "SELECTION_REASON"]].head()

,RSN,EVENT,SCORE,ERROR_TOTAL,PARETO_RANK,SELECTION_REASON
20,1160,"Kocaeli, Turkey",89.984734,0.111300,0,selected
22,1163,"Kocaeli, Turkey",88.782399,0.126349,0,selected
24,1177,"Kocaeli, Turkey",88.410961,0.131081,0,selected
16,900,Landers,86.757991,0.152632,0,selected
27,1636,"Manjil, Iran",86.482139,0.156308,0,selected


## 10. Common PEER/AFAD output shape

Both providers are normalized to shared columns. This cell shows the selected PEER columns and the expected columns that AFAD rows also use after mapping.

In [19]:
common_columns = [
    "PROVIDER",
    "RSN",
    "EVENT",
    "YEAR",
    "MAGNITUDE",
    "STATION",
    "VS30(m/s)",
    "RJB(km)",
    "RRUP(km)",
    "MECHANISM",
    "PGA(cm2/sec)",
    "PGV(cm/sec)",
    "PGD(cm)",
    "T90_avg(sec)",
    "ARIAS_INTENSITY(m/sec)",
    "ENDPOINTSOURCE",
    "FILE_NAME_H1",
]

available_common_columns = [column for column in common_columns if column in selected.columns]
selected[available_common_columns].head()

,PROVIDER,RSN,EVENT,YEAR,MAGNITUDE,STATION,VS30(m/s),RJB(km),RRUP(km),MECHANISM,PGA(cm2/sec),PGV(cm/sec),PGD(cm),T90_avg(sec),ARIAS_INTENSITY(m/sec),ENDPOINTSOURCE,FILE_NAME_H1
22,PEER,1163,"Kocaeli, Turkey",1999,7.51,Hava Alani,354.37,58.33,60.05,StrikeSlip,87.990167,19.000,23.2830,36.7,0.2,None,KOCAELI\DHM000.AT2
24,PEER,1177,"Kocaeli, Turkey",1999,7.51,Zeytinburnu,341.56,51.98,53.88,StrikeSlip,106.666932,15.551,18.2290,39.4,0.3,None,KOCAELI\ZYT000.AT2
19,PEER,1157,"Kocaeli, Turkey",1999,7.51,Cekmece,346.00,64.95,66.69,StrikeSlip,156.053221,12.931,15.9030,37.0,0.5,None,KOCAELI\CNA000.AT2
16,PEER,900,Landers,1992,7.28,Yermo Fire Station,353.63,23.62,23.62,StrikeSlip,217.776277,40.263,33.1550,18.9,0.9,None,LANDERS\YER270.AT2
10,PEER,862,Landers,1992,7.28,Indio - Coachella Canal,339.02,54.25,54.25,StrikeSlip,102.185293,13.370,6.9545,37.9,0.3,None,LANDERS\IND000.AT2


## 11. Export selected records and report

This mirrors the CLI flow: selected CSV plus JSON report.

In [20]:
output_dir = Path("example_outputs")
output_dir.mkdir(exist_ok=True)

selected_csv = output_dir / "selected_constraint_records.csv"
report_json = output_dir / "constraint_report.json"

selected.to_csv(selected_csv, index=False)
report_json.write_text(
    json.dumps(report, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)

{
    "selected_csv": str(selected_csv.resolve()),
    "report_json": str(report_json.resolve()),
}

{'selected_csv': 'D:\\github\\SelectionEarthquake\\examples\\example_outputs\\selected_constraint_records.csv',
 'report_json': 'D:\\github\\SelectionEarthquake\\examples\\example_outputs\\constraint_report.json'}

## 12. CLI equivalent

The same workflow can be executed from a shell after installation.

In [21]:
print(
    "quake-sel "
    "--providers peer "
    "--strategy constraint "
    "--num-records 11 "
    "--report-path selection_report.json "
    "--selected-csv selected_records.csv"
)

quake-sel --providers peer --strategy constraint --num-records 11 --report-path selection_report.json --selected-csv selected_records.csv


## 13. Optional AFAD search and waveform download

Set `RUN_AFAD = True` only when you have network access and want to call the live AFAD service. PEER waveform download is intentionally skipped by the library because PEER is a local flatfile provider here.

In [22]:
RUN_AFAD = False

if RUN_AFAD:
    afad_criteria = SearchCriteria(
        start_date="2023-02-06",
        end_date="2023-02-07",
        min_magnitude=6.0,
        max_magnitude=8.0,
        mechanisms=["Reverse"],
        weights=ScoringWeights.from_preset("tbdy_2018_record_selection"),
    )
    afad_strategy = TBDY2018ConstraintStrategy(config=config)
    afad_api = EarthquakeAPI(
        provider_names=[ProviderName.AFAD],
        strategies=[afad_strategy],
        use_cache=True,
    )
    afad_result = afad_api.run_sync(afad_criteria, afad_strategy.get_name())
    if not afad_result.success:
        raise RuntimeError(afad_result.error)
    display(afad_result.value.selected_df.head())
else:
    print("AFAD live search skipped. Set RUN_AFAD = True to run it.")

AFAD live search skipped. Set RUN_AFAD = True to run it.


In [23]:
if RUN_AFAD:
    download_result = afad_api.download_waveforms(
        afad_result.value.selected_df,
        export_type="mseed",
    )
    print(download_result)
else:
    print("AFAD waveform download skipped. It requires RUN_AFAD = True and live AFAD rows.")

AFAD waveform download skipped. It requires RUN_AFAD = True and live AFAD rows.


## 14. What to inspect next

- `result.selected_df`: selected records only.
- `result.scored_df`: all considered records with `SCORE`, `SELECTION_STATUS`, and `SELECTION_REASON`.
- `SCORE_BREAKDOWN`: weighted Gaussian contribution by criterion.
- `HARD_FILTERS`, `ERROR_METRICS`, `ERROR_TOTAL`: constraint-first traceability.
- `PARETO_RANK`, `PARETO_FRONT`: Pareto strategy diagnostics.
- `SPECTRUM_ERROR`: spectrum/intensity proxy strategy diagnostic.
- `result.report`: JSON-safe report payload for UI, CLI, or external reporting.